# NAM Chain Capture — PoC no Colab

Encadeia N capturas `.nam` + IRs e treina **um único `.nam`** que reproduz a cadeia inteira.

1. `Runtime > Change runtime type > GPU` (T4 basta).
2. Rode as células em ordem.
3. No fim você baixa `chain.a1.nam` (formato legado 0.5.x, para pedaleiras) e `chain.nam` (formato atual), mais o `report.json` com ESR, tempo e custo.

import os
if not os.path.exists("nam-chain-capture"):
    !git clone --depth 1 https://github.com/Longhin08/nam-chain-capture.git
%cd nam-chain-capture/poc
!pip install -q -r requirements.txt

In [ ]:
import os
if not os.path.exists("nam-chain-capture"):
    !git clone --depth 1 https://github.com/Longhin08/nam-chain-capture.git
%cd nam-chain-capture/poc
!pip install -q -r requirements.txt

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NENHUMA — troque o runtime!")

## 2. Sinal de treino oficial (v3_0_0.wav)

In [ ]:
!mkdir -p data
!test -f data/v3_0_0.wav || curl -sSL -o data/v3_0_0.wav "https://drive.usercontent.google.com/download?id=1Pgf8PdE0rKB1TD4TRPKbpNo1ByR3IOm9&export=download&confirm=t"
!md5sum data/v3_0_0.wav   # esperado: 36cd1af62985c2fac3e654333e36431e

## 3. Envie suas capturas (`.nam`) e IRs (`.wav`)

In [ ]:
from google.colab import files
up = files.upload()
for name, data in up.items():
    open(f"data/{name}", "wb").write(data)
print(sorted(os.listdir("data")))

## 4. Monte a cadeia
A ordem da lista é a ordem do sinal (drive → amp → cab). Ganhos em dB. `ir_max_ms: 20` trunca o IR (IRs longos perdem cauda no modelo final).

In [ ]:
import json
chain = {
    "name": "Meu rig",
    "arch": "standard",        # standard | lite | feather | nano  (use o máximo que seu pedal aguenta)
    "stages": [
        {"type": "nam", "file": "drive.nam", "input_gain_db": 0, "output_gain_db": 0},
        {"type": "nam", "file": "amp.nam"},
        {"type": "ir",  "file": "cab.wav", "ir_max_ms": 20},
    ],
}
json.dump(chain, open("data/chain.json", "w"), indent=2)

## 5. (Opcional) Ouça a cadeia antes de gastar GPU

In [ ]:
!python -m chaincap render data/chain.json --input data/v3_0_0.wav --out out
from IPython.display import Audio
from scipy.io import wavfile
sr, y = wavfile.read("out/target.wav")
Audio(y[48000*20:48000*35].astype("float32") / 2**31, rate=sr)   # trecho de 15 s

## 6. Treinar
`--gpu-usd-per-hour` só serve para estimar o custo no relatório (T4 spot ≈ 0,35 USD/h).

In [ ]:
!python -m chaincap run data/chain.json --input data/v3_0_0.wav --out out --epochs 100 --gpu-usd-per-hour 0.35

## 7. Baixar resultado

In [ ]:
for f in ["out/chain.a1.nam", "out/chain.nam", "out/report.json"]:
    if os.path.exists(f):
        files.download(f)